In [1]:
!pip install selenium webdriver-manager beautifulsoup4 lxml pandas tqdm
!apt-get update
!apt install -y chromium-chromedriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntu

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5aa40d230b9a <unknown>
#1 0x5aa40cc32265 <unknown>
#2 0x5aa40cc6ef3b <unknown>
#3 0x5aa40cc69c09 <unknown>
#4 0x5aa40ccb9f9e <unknown>
#5 0x5aa40ccb968c <unknown>
#6 0x5aa40cc785cf <unknown>
#7 0x5aa40cc79391 <unknown>
#8 0x5aa40d1f607b <unknown>
#9 0x5aa40d1f903d <unknown>
#10 0x5aa40d1e2838 <unknown>
#11 0x5aa40d1f9bd0 <unknown>
#12 0x5aa40d1c92b0 <unknown>
#13 0x5aa40d21dde8 <unknown>
#14 0x5aa40d21dfb8 <unknown>
#15 0x5aa40d22f60e <unknown>
#16 0x7e4c4b4cfac3 <unknown>


In [ ]:
import pandas as pd
import time
from bs4 import BeautifulSoup
from tqdm import tqdm
from selenium.webdriver.common.by import By

def get_jobs(keyword, num_jobs=20):
    jobs = []

    url = f"https://www.glassdoor.com/Job/jobs.htm?sc.keyword={keyword}"
    driver.get(url)

    time.sleep(5)

    while len(jobs) < num_jobs:
        job_buttons = driver.find_elements(By.CLASS_NAME, "jl")

        for job_button in job_buttons:
            if len(jobs) >= num_jobs:
                break

            try:
                job_button.click()
                time.sleep(2)

                company_name = driver.find_element(By.CLASS_NAME, "employerName").text
                location = driver.find_element(By.CLASS_NAME, "location").text
                job_title = driver.find_element(By.CLASS_NAME, "title").text

                jobs.append({
                    "Job Title": job_title,
                    "Company Name": company_name,
                    "Location": location
                })

            except Exception as e:
                print("Skipping:", e)

        break

    return pd.DataFrame(jobs)

In [ ]:
df = get_jobs("data scientist", 20)
df.head()

In [ ]:
driver.find_element(By.XPATH, "xpath")
driver.find_element(By.CLASS_NAME, "class")

In [ ]:
df.to_csv("jd_unstructured_data.csv", index=False)
print("CSV file created")

In [ ]:
!pip install webdriver-manager

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
stopw  = set(stopwords.words('english'))

# Load the dataset:
unstructured_df=pd.read_csv('jd_unstructured_data.csv')

def convert_salary(value):
    if 'Unknown' in value:
        return None
    elif '-' in value:
        values = re.findall(r'\$\d+K', value)
        min_value = int(values[0].replace('$', '').replace('K', '')) if values else None
        max_value = int(values[1].replace('$', '').replace('K', '')) if len(values) > 1 else None
        if min_value and max_value:
            return (min_value + max_value) / 2
        elif min_value:
            return min_value
        elif max_value:
            return max_value
        else:
            return None
    else:
        return int(re.findall(r'\$\d+K', value)[0].replace('$', '').replace('K', ''))

def convert_revenue(value):
    if 'Unknown' in value:
        return None
    elif ' to ' in value:
        values = re.findall(r'\d+\.?\d*', value)
        min_revenue = float(values[0])
        max_revenue = float(values[1])
        unit = value.split()[-2]
        if unit == 'billion':
            min_revenue *= 1000
            max_revenue *= 1000
        return (min_revenue + max_revenue) / 2
    else:
        numerical_values = re.findall(r'\d+\.?\d*', value)
        if numerical_values:
            return float(numerical_values[0])
        else:
            return None

# Define a function to convert the size value
def convert_size(value):
    if 'Unknown' in value:
        return None
    elif ' to ' in value:
        sizes = value.split(' to ')
        min_size = int(sizes[0].replace('+', '').replace(',', '').split()[0])
        max_size = int(sizes[1].replace('+', '').replace(',', '').split()[0])
        return (min_size + max_size) / 2
    else:
        return int(value.replace('+', '').replace(',', '').split()[0])

# Apply the conversion function to the "Salary Column" column
unstructured_df['Average Salary'] = unstructured_df['Salary Estimate'].apply(convert_salary)

# Apply the conversion function to the "Revenue" column
unstructured_df['Average Revenue'] = unstructured_df['Revenue'].apply(convert_revenue)

# Extract the company name by splitting on '\r\n' and selecting the first element
unstructured_df['Company Name'] = unstructured_df['Company Name'].str.split('\r\n').str[0]


# Apply the conversion function to the "Size" column
unstructured_df['Size'] = unstructured_df['Size'].apply(convert_size)

# remove stopwords and pre-process Job Description Column:
unstructured_df['Processed_JD']=unstructured_df['Job Description'].apply(lambda x: ' '.join([word for word in str(x).split() if len(word)>2 and word not in (stopw)]))


# Drop Unwanted Columns:
unstructured_df=unstructured_df.drop(['Unnamed: 0','Salary Estimate','Revenue','Job Description'],axis=1)

# Check for Null Value after data pre-processing:
unstructured_df.isnull().sum()

# Replace the null values with average value of each columns:
# Calculate the average value of column B
size_average = unstructured_df['Size'].mean()
salary_average=unstructured_df['Average Salary'].mean()
revenue_average=unstructured_df['Average Revenue'].mean()

# Replace null values with the average
unstructured_df['Size'].fillna(size_average, inplace=True)
unstructured_df['Average Salary'].fillna(salary_average, inplace=True)
unstructured_df['Average Revenue'].fillna(revenue_average, inplace=True)

# Convert DataFrame to CSV file
unstructured_df.to_csv(r'C:\Users\Admin\ML_Projects\Job_Recommendation_System\Job-Recommendation-System\src\data\jd_structured_data.csv', index=False)

In [ ]:
import re
from ftfy import fix_text
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
stopw  = set(stopwords.words('english'))
from pyresparser import ResumeParser
import os
from docx import Document
import src.notebook.skills_extraction as skills_extraction

# Load dataset:
jd_df=pd.read_csv(r'C:\Users\Admin\ML_Projects\Job_Recommendation_System\Job-Recommendation-System\src\data\jd_structured_data.csv')

# Load the extracted resume skills:
file_path=r'C:\Users\Admin\ML_Projects\Job_Recommendation_System\Job-Recommendation-System\utilities\resumes\CV.pdf'
skills=[]
skills.append(' '.join(word for word in skills_extraction.skills_extractor(file_path)))

def ngrams(string, n=3):
    string = fix_text(string) # fix text
    string = string.encode("ascii", errors="ignore").decode() #remove non ascii chars
    string = string.lower()
    chars_to_remove = [")","(",".","|","[","]","{","}","'"]
    rx = '[' + re.escape(''.join(chars_to_remove)) + ']'
    string = re.sub(rx, '', string)
    string = string.replace('&', 'and')
    string = string.replace(',', ' ')
    string = string.replace('-', ' ')
    string = string.title() # normalise case - capital at start of each word
    string = re.sub(' +',' ',string).strip() # get rid of multiple spaces and replace with a single
    string = ' '+ string +' ' # pad names for ngrams...
    string = re.sub(r'[,-./]|\sBD',r'', string)
    ngrams = zip(*[string[i:] for i in range(n)])
    return [''.join(ngram) for ngram in ngrams]

vectorizer = TfidfVectorizer(min_df=1, analyzer=ngrams, lowercase=False)
tfidf = vectorizer.fit_transform(skills)

nbrs = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(tfidf)
jd_test = (jd_df['Processed_JD'].values.astype('U'))

def getNearestN(query):
  queryTFIDF_ = vectorizer.transform(query)
  distances, indices = nbrs.kneighbors(queryTFIDF_)
  return distances, indices

distances, indices = getNearestN(jd_test)
test = list(jd_test)
matches = []

for i,j in enumerate(indices):
    dist=round(distances[i][0],2)

    temp = [dist]
    matches.append(temp)

matches = pd.DataFrame(matches, columns=['Match confidence'])

# Following recommends Top 5 Jobs based on the candidate resume:
jd_df['match']=matches['Match confidence']
jd_df.head(5).sort_values('match')

In [ ]:
import spacy
from spacy.matcher import Matcher
import PyPDF2
import os

# Load the Spacy English model
nlp = spacy.load('en_core_web_sm')
import csv
from spacy.matcher import Matcher
import csv

# Read skills from CSV file
file_path=r'C:\Users\Admin\ML_Projects\Job_Recommendation_System\Job-Recommendation-System\src\data\skills.csv'
with open(file_path, 'r') as file:
    csv_reader = csv.reader(file)
    skills = [row for row in csv_reader]

# Create pattern dictionaries from skills
skill_patterns = [[{'LOWER': skill}] for skill in skills[0]]

# Create a Matcher object
matcher = Matcher(nlp.vocab)

# Add skill patterns to the matcher
for pattern in skill_patterns:
    matcher.add('Skills', [pattern])

# Function to extract skills from text
def extract_skills(text):
    doc = nlp(text)
    matches = matcher(doc)
    skills = set()
    for match_id, start, end in matches:
        skill = doc[start:end].text
        skills.add(skill)
    return skills

# Function to extract text from PDF
def extract_text_from_pdf(file_path:str):
    with open(file_path, 'rb') as f:
        pdf_reader = PyPDF2.PdfReader(f)
        text = ''
        for page in pdf_reader.pages:
            text += page.extract_text()
    return text

def skills_extractor(file_path):
        # Extract text from PDF
        path=r'C:\Users\Admin\ML_Projects\Job_Recommendation_System\Job-Recommendation-System\src\notebook'
        full_file_path = os.path.join(path, file_path)
        resume_text = extract_text_from_pdf(full_file_path)

        # Extract skills from resume text
        skills = list(extract_skills(resume_text))
        return skills

In [ ]:
import streamlit as st
import pandas as pd
import PyPDF2
from pyresparser import ResumeParser
from sklearn.neighbors import NearestNeighbors
from src.components.job_recommender import ngrams,getNearestN,jd_df
import src.notebook.skills_extraction as skills_extraction
from sklearn.feature_extraction.text import TfidfVectorizer


# Function to process the resume and recommend jobs
def process_resume(file_path):
    # Extract text from PDF resume
    resume_skills=skills_extraction.skills_extractor(file_path)

    # Perform job recommendation based on parsed resume data
    skills=[]
    skills.append(' '.join(word for word in resume_skills))


    # Feature Engineering:
    vectorizer = TfidfVectorizer(min_df=1, analyzer=ngrams, lowercase=False)
    tfidf = vectorizer.fit_transform(skills)


    nbrs = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(tfidf)
    jd_test = (jd_df['Processed_JD'].values.astype('U'))

    distances, indices = getNearestN(jd_test)
    test = list(jd_test)
    matches = []

    for i,j in enumerate(indices):
        dist=round(distances[i][0],2)
        temp = [dist]
        matches.append(temp)

    matches = pd.DataFrame(matches, columns=['Match confidence'])

    # Following recommends Top 5 Jobs based on candidate resume:
    jd_df['match']=matches['Match confidence']

    return jd_df.head(5).sort_values('match')

# Streamlit app
def main():
    st.title("Job Recommendation App")
    st.write("Upload your resume in PDF format")

    # File uploader
    uploaded_file = st.file_uploader("Choose a file", type=['pdf'])

    if uploaded_file is not None:
        # Process resume and recommend jobs
        file_path=uploaded_file.name
        df_jobs = process_resume(file_path)

        # Display recommended jobs as DataFrame
        st.write("Recommended Jobs:")
        st.dataframe(df_jobs[['Job Title','Company Name','Location','Industry','Sector','Average Salary']])

# Run the Streamlit app
if __name__ == '__main__':
    main()